# EDA â€” NWB (Nationaal Wegenbestand) Dataset

This notebook explores the raw NWB data to build intuition about:
- What columns and road types the dataset contains
- How junctions are encoded in the NWB (the `JTE_ID_BEG` / `JTE_ID_END` structure)
- How the current intersection selection pipeline works, step by step
- What the `street_count` actually measures and its distribution
- What changes when you alter the `MIN_CONNECTIONS` threshold
- Potential issues with the current selection (dual carriageways, ramps, etc.)

**Input files:**
- `data/raw/Wegvakken.gpkg` â€” NWB road segments for the whole Netherlands
- `data/raw/rotterdam_ring.geojson` â€” area of interest polygon
- `data/processed/intersections.gpkg` â€” intersections already extracted by notebook 00

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
from shapely.geometry import Point

# Base paths — same as other notebooks
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

WEGVAKKEN_PATH     = os.path.join(PROJECT_DIR, "data", "raw",       "Wegvakken.gpkg")
RING_PATH          = os.path.join(PROJECT_DIR, "data", "raw",       "rotterdam_ring.geojson")
INTERSECTIONS_PATH = os.path.join(PROJECT_DIR, "data", "processed", "intersections.gpkg")

CRS_RD = "EPSG:28992"  # RD New — all NWB data is already in this projection

# Road types that the pipeline includes — only normal roads.
# Excludes: NRB (side carriageways), AFR (off-ramps), OPR (on-ramps), FP, parking.
INCLUDE_BST = {"RB", "ERF", "HR"}

# Road manager filter: only gemeente-managed roads (excludes Rijk highways, Provincie).
# WEGBEHSRT: R=Rijk, P=Provincie, G=Gemeente, W=Waterschap.
WEGBEHSRT_GEMEENTE = "G"

print("Setup complete.")


---
## 1. What does the raw Wegvakken file actually look like?

Before anything else, let's load the full nationwide NWB and inspect its columns.
This reveals what information is available per road segment beyond just geometry.

In [ ]:
# Load the full nationwide NWB â€” takes a moment (~1.6M rows)
print("Loading wegvakken...")
wegvakken_nl = gpd.read_file(WEGVAKKEN_PATH)
print(f"Total nationwide rows: {len(wegvakken_nl):,}")

# Show all columns and their data types
print("\nColumns and dtypes:")
print(wegvakken_nl.dtypes)

In [ ]:
# Preview a few rows to see the actual content
wegvakken_nl.drop(columns="geometry").head(5)

In [ ]:
# How many unique municipalities are in the dataset?
# This gives a sense of the nationwide scope.
n_municipalities = wegvakken_nl["GME_NAAM"].nunique()
print(f"Unique municipalities (GME_NAAM): {n_municipalities:,}")

# How many segments does Rotterdam have compared to the national total?
rotterdam_count = (wegvakken_nl["GME_NAAM"] == "Rotterdam").sum()
print(f"Rotterdam share: {rotterdam_count:,} / {len(wegvakken_nl):,} ({rotterdam_count/len(wegvakken_nl)*100:.1f}%)")

---
## 2. What road types (BST_CODE) exist, and which do we include?

`BST_CODE` is the field that classifies every road segment into a functional type.
The current pipeline keeps only certain types to exclude footpaths, cycleways, and parking.

Here we look at the full distribution â€” nationwide and Rotterdam â€” to understand what
is being included vs. excluded and why.

In [ ]:
# Full BST_CODE distribution nationwide â€” see all possible road type codes
bst_national = wegvakken_nl["BST_CODE"].value_counts()
print("BST_CODE distribution (nationwide):")
print(bst_national.to_string())

In [ ]:
# Filter to Rotterdam and show distribution there
wegvakken_rot = wegvakken_nl[wegvakken_nl["GME_NAAM"] == "Rotterdam"].copy()
bst_rotterdam = wegvakken_rot["BST_CODE"].value_counts()

# Flag which codes are included vs excluded by the current pipeline
bst_df = bst_rotterdam.reset_index()
bst_df.columns = ["BST_CODE", "count"]
bst_df["included_by_pipeline"] = bst_df["BST_CODE"].isin(INCLUDE_BST)
bst_df["pct_of_rotterdam"] = (bst_df["count"] / len(wegvakken_rot) * 100).round(1)

print("BST_CODE distribution â€” Rotterdam:")
print(bst_df.to_string(index=False))

In [ ]:
# Visual: bar chart of BST_CODE with include/exclude colouring
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["steelblue" if inc else "lightcoral" for inc in bst_df["included_by_pipeline"]]
bars = ax.bar(bst_df["BST_CODE"], bst_df["count"], color=colors, edgecolor="white")

# Add count labels on bars
for bar, count in zip(bars, bst_df["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha="center", va="bottom", fontsize=9)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="steelblue", label="Included (INCLUDE_BST)"),
    Patch(facecolor="lightcoral", label="Excluded")
])

ax.set_xlabel("BST_CODE")
ax.set_ylabel("Number of road segments")
ax.set_title("Road type distribution in Rotterdam â€” which types are included?")
plt.tight_layout()
plt.show()

# Summarise totals
n_included = bst_df.loc[bst_df["included_by_pipeline"], "count"].sum()
n_excluded = bst_df.loc[~bst_df["included_by_pipeline"], "count"].sum()
print(f"Included segments: {n_included:,} ({n_included/len(wegvakken_rot)*100:.1f}%)")
print(f"Excluded segments: {n_excluded:,} ({n_excluded/len(wegvakken_rot)*100:.1f}%)")

---
## 3. How does the NWB encode junctions?

This is the core concept behind the intersection extraction pipeline.

Every NWB road segment (`wegvak`) has:
- `JTE_ID_BEG` â€” the junction ID at the **start** of the linestring
- `JTE_ID_END` â€” the junction ID at the **end** of the linestring

A 'junction' only becomes a true intersection when **3 or more** road segments connect to it.
The cell below visualises this with a concrete example.

In [ ]:
# Filter to gemeente-managed roads first — removes Rijk highways and provincial roads
# that would otherwise inflate junction counts at motorway on/off-ramps
wegvakken = wegvakken_rot[wegvakken_rot["WEGBEHSRT"] == WEGBEHSRT_GEMEENTE].copy()
print(f"Wegvakken after WEGBEHSRT='G' filter: {len(wegvakken):,} segments")

# Filter to the road types used by the pipeline (normal roads only)
wegvakken = wegvakken[wegvakken["BST_CODE"].isin(INCLUDE_BST)].copy()
print(f"Wegvakken after BST_CODE filter: {len(wegvakken):,} segments")

# How many unique junction IDs appear as start vs end points?
n_beg = wegvakken["JTE_ID_BEG"].nunique()
n_end = wegvakken["JTE_ID_END"].nunique()
all_ids = pd.Index(wegvakken["JTE_ID_BEG"]).union(pd.Index(wegvakken["JTE_ID_END"]))
print(f"\nUnique JTE_ID_BEG values: {n_beg:,}")
print(f"Unique JTE_ID_END values: {n_end:,}")
print(f"Total unique junction IDs (union): {len(all_ids):,}")


In [ ]:
# Are any junction IDs ONLY ever a start point, or ONLY ever an end point?
# A junction that only appears as BEG or only as END might be a dead-end or data artifact.
beg_only = set(wegvakken["JTE_ID_BEG"]) - set(wegvakken["JTE_ID_END"])
end_only  = set(wegvakken["JTE_ID_END"]) - set(wegvakken["JTE_ID_BEG"])
both      = set(wegvakken["JTE_ID_BEG"]) & set(wegvakken["JTE_ID_END"])

print(f"Junctions that ONLY appear as BEG: {len(beg_only):,}")
print(f"Junctions that ONLY appear as END: {len(end_only):,}")
print(f"Junctions that appear as both:     {len(both):,}")
print()
print("Note: 'BEG-only' and 'END-only' junctions are likely dead-ends (cul-de-sacs)")
print("or the edge of our municipality filter. They won't become intersections (street_count < 3).")

In [ ]:
# Visualise the concept: pick one known intersection and draw its connected wegvakken
# Load the saved intersections to pick a concrete example
intersections = gpd.read_file(INTERSECTIONS_PATH).set_index("JTE_ID")

# Pick an intersection with exactly 4 legs (classic crossroads)
example_4leg = intersections[intersections["street_count"] == 4].index[0]
ix = intersections.loc[example_4leg].geometry.x
iy = intersections.loc[example_4leg].geometry.y

# Find all wegvakken that connect to this junction
connected_wvk = wegvakken[
    (wegvakken["JTE_ID_BEG"] == example_4leg) |
    (wegvakken["JTE_ID_END"] == example_4leg)
]

print(f"Example intersection (JTE_ID={example_4leg}):")
print(f"  Location: ({ix:.0f}, {iy:.0f}) RD New")
print(f"  street_count from intersections.gpkg: {intersections.loc[example_4leg, 'street_count']}")
print(f"  Connected wegvakken found: {len(connected_wvk)}")
print(connected_wvk[["WVK_ID", "JTE_ID_BEG", "JTE_ID_END", "BST_CODE"]].to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 8))

# Buffer around intersection to limit what we show
buffer = 200
nearby_wvk = wegvakken.cx[ix - buffer:ix + buffer, iy - buffer:iy + buffer]

nearby_wvk.plot(ax=ax, color="lightgrey", linewidth=1, label="Other road segments")
connected_wvk.plot(ax=ax, color="steelblue", linewidth=3, label="Connected to junction")
ax.scatter(ix, iy, color="red", s=150, zorder=5, label=f"Junction {example_4leg}")

# Label the other junction IDs at the far ends of connected wegvakken
for _, wvk in connected_wvk.iterrows():
    if wvk["JTE_ID_BEG"] == example_4leg:
        nx, ny = wvk.geometry.coords[-1]
        label  = str(wvk["JTE_ID_END"])
    else:
        nx, ny = wvk.geometry.coords[0]
        label  = str(wvk["JTE_ID_BEG"])
    ax.scatter(nx, ny, color="orange", s=60, zorder=4)
    ax.annotate(f"JTE {label[-5:]}", (nx, ny), fontsize=7, ha="center", va="bottom",
                color="darkorange")

ax.set_xlim(ix - buffer, ix + buffer)
ax.set_ylim(iy - buffer, iy + buffer)
ax.set_aspect("equal")
ax.legend()
ax.set_title(f"How NWB encodes an intersection\nJTE_ID={example_4leg}, street_count=4")
plt.tight_layout()
plt.show()

---
## 4. How is `street_count` calculated, and what is its distribution?

The pipeline derives `street_count` by summing how many road segments reference each junction ID
as either their `JTE_ID_BEG` or `JTE_ID_END`. This is effectively the degree of the node in the road network graph.

Understanding this distribution helps explain why `MIN_CONNECTIONS = 3` was chosen.

In [ ]:
# Reproduce the street_count calculation from notebook 00 to inspect the full distribution
# (including junctions with < 3 connections that get filtered out)
beg_counts = wegvakken["JTE_ID_BEG"].value_counts()
end_counts  = wegvakken["JTE_ID_END"].value_counts()
connection_counts = beg_counts.add(end_counts, fill_value=0).astype(int)
connection_counts.name = "street_count"

print("street_count distribution (ALL Rotterdam junctions, before any threshold filter):")
full_dist = connection_counts.value_counts().sort_index()
print(full_dist.to_string())
print(f"\nTotal junctions: {len(connection_counts):,}")

In [ ]:
# Plot the full distribution with an annotation showing where the threshold cuts
fig, ax = plt.subplots(figsize=(10, 5))

# Clip x-axis at 10 for readability â€” very high counts are rare
dist_clipped = connection_counts.clip(upper=10).value_counts().sort_index()
colors_bar = ["lightcoral" if sc < 3 else "steelblue" for sc in dist_clipped.index]

bars = ax.bar(dist_clipped.index.astype(str), dist_clipped.values,
              color=colors_bar, edgecolor="white")

# Label bars
for bar, count in zip(bars, dist_clipped.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{count:,}", ha="center", va="bottom", fontsize=8)

# Threshold line
ax.axvline(x=1.5, color="black", linestyle="--", linewidth=1.5,
           label="MIN_CONNECTIONS=3 threshold")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="lightcoral", label="Excluded (dead-ends / pass-through points)"),
    Patch(facecolor="steelblue",  label="Included as intersection"),
])

ax.set_xlabel("street_count (road segments connecting at this junction)")
ax.set_ylabel("Number of junctions")
ax.set_title("Junction degree distribution â€” Rotterdam (BST_CODE filtered)\nValues capped at 10 for readability")
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the threshold effect â€” how many junctions make it through at different cut-offs?
thresholds = [2, 3, 4, 5]
print(f"{'threshold':>10}  {'junctions':>12}  {'% of all':>10}")
print("-" * 38)
for t in thresholds:
    n = (connection_counts >= t).sum()
    print(f"{t:>10}  {n:>12,}  {n/len(connection_counts)*100:>9.1f}%")

In [ ]:
# What about the intersections that PASSED the threshold â€” are they in the ring?
# Clip to the ring (same as notebook 00) and check how much the ring reduces the count.
ring = gpd.read_file(RING_PATH).to_crs(CRS_RD)
ring_polygon = ring.union_all()

# Build GeoDataFrame of all junctions with street_count >= 3 (before ring clip)
starts = wegvakken[["JTE_ID_BEG", "geometry"]].copy()
starts["point"] = starts["geometry"].apply(lambda g: Point(g.coords[0]))
starts = starts.rename(columns={"JTE_ID_BEG": "JTE_ID"})[["JTE_ID", "point"]]

ends = wegvakken[["JTE_ID_END", "geometry"]].copy()
ends["point"] = ends["geometry"].apply(lambda g: Point(g.coords[-1]))
ends = ends.rename(columns={"JTE_ID_END": "JTE_ID"})[["JTE_ID", "point"]]

all_junctions = pd.concat([starts, ends], ignore_index=True).drop_duplicates(subset="JTE_ID")
all_junctions = all_junctions.set_index("JTE_ID")
all_junctions["street_count"] = connection_counts
all_junctions = all_junctions[all_junctions["street_count"] >= 3].reset_index()

inter_gdf = gpd.GeoDataFrame(all_junctions, geometry="point", crs=CRS_RD)
inter_in_ring  = inter_gdf[inter_gdf.within(ring_polygon)]
inter_out_ring = inter_gdf[~inter_gdf.within(ring_polygon)]

print(f"Junctions with street_count >= 3 in Rotterdam: {len(inter_gdf):,}")
print(f"  Inside ring:  {len(inter_in_ring):,} ({len(inter_in_ring)/len(inter_gdf)*100:.1f}%)")
print(f"  Outside ring: {len(inter_out_ring):,} ({len(inter_out_ring)/len(inter_gdf)*100:.1f}%)")

---
## 5. Spatial distribution â€” where are the intersections?

A map comparison of intersections inside vs outside the ring,
and a density heat-map to show clustering.

In [ ]:
# Map: all Rotterdam intersections coloured by inside/outside ring
fig, ax = plt.subplots(figsize=(14, 14))

# Plot the road network for context (thin grey lines)
wegvakken.plot(ax=ax, color="lightgrey", linewidth=0.3, alpha=0.6)

# All intersections outside ring
inter_out_ring.plot(ax=ax, color="lightcoral", markersize=2, alpha=0.5, label="Outside ring")

# All intersections inside ring
inter_in_ring.plot(ax=ax, color="steelblue", markersize=4, alpha=0.8, label="Inside ring (used)")

# Ring boundary
ring.plot(ax=ax, color="none", edgecolor="red", linewidth=2, label="Rotterdam ring")

ax.set_aspect("equal")
ax.legend(markerscale=3)
ax.set_title(f"NWB intersections in Rotterdam â€” {len(inter_in_ring):,} inside ring, "
             f"{len(inter_out_ring):,} outside")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of street_count for the intersections INSIDE the ring
sc_dist = intersections["street_count"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sc_dist.index.astype(str), sc_dist.values, color="steelblue", edgecolor="white")
for x, count in zip(sc_dist.index, sc_dist.values):
    ax.text(str(x), count + 5, str(count), ha="center", va="bottom", fontsize=9)

ax.set_xlabel("street_count")
ax.set_ylabel("Number of intersections")
ax.set_title("Distribution of street_count â€” intersections inside ring")
plt.tight_layout()
plt.show()

print("Summary:")
print(intersections["street_count"].describe())

---
## 6. How does the pipeline derive leg bearings from NWB?

Each intersection has multiple 'legs' â€” the approach roads connecting to it.
The pipeline extracts these directly from the NWB geometry instead of using an OSM graph.

For each wegvak that touches a junction, the bearing from the **other** endpoint toward
the junction gives the approach bearing for that leg.

This section traces through the logic for a few concrete examples.

In [ ]:
def bearing_between(x1, y1, x2, y2):
    """Compass bearing from (x1,y1) to (x2,y2) in RD New coordinates.
    Returns 0-360Â°, where 0=North and 90=East (clockwise convention)."""
    dx = x2 - x1
    dy = y2 - y1
    return np.degrees(np.arctan2(dx, dy)) % 360


def get_leg_bearings_verbose(intersection_jte_id, wegvakken_gdf, intersections_gdf):
    """Reproduce the leg extraction from notebook 04, but with verbose output.
    Returns a DataFrame showing every step for one intersection."""
    node = intersections_gdf.loc[intersection_jte_id]
    ix, iy = node.geometry.x, node.geometry.y

    # All wegvakken that start OR end at this junction
    connected = wegvakken_gdf[
        (wegvakken_gdf["JTE_ID_BEG"] == intersection_jte_id) |
        (wegvakken_gdf["JTE_ID_END"] == intersection_jte_id)
    ]

    rows = []
    seen_bearings = []

    for _, wvk in connected.iterrows():
        # Determine which end is this junction and which end is the neighbor
        if wvk["JTE_ID_BEG"] == intersection_jte_id:
            neighbor_coord = wvk.geometry.coords[-1]   # intersection at BEG â†’ neighbor at END
            neighbor_jte   = wvk["JTE_ID_END"]
            junction_role  = "BEG"
        else:
            neighbor_coord = wvk.geometry.coords[0]    # intersection at END â†’ neighbor at BEG
            neighbor_jte   = wvk["JTE_ID_BEG"]
            junction_role  = "END"

        nx, ny = neighbor_coord[0], neighbor_coord[1]
        approach_bearing = bearing_between(nx, ny, ix, iy)

        # Check if this bearing is a near-duplicate of one already seen
        is_dup = any(abs((approach_bearing - b + 180) % 360 - 180) < 20 for b in seen_bearings)

        rows.append({
            "WVK_ID":          wvk["WVK_ID"],
            "BST_CODE":        wvk["BST_CODE"],
            "junction_role":   junction_role,
            "neighbor_jte_id": neighbor_jte,
            "neighbor_x":      round(nx, 1),
            "neighbor_y":      round(ny, 1),
            "approach_bearing": round(approach_bearing, 1),
            "is_duplicate":    is_dup
        })

        if not is_dup:
            seen_bearings.append(approach_bearing)

    return pd.DataFrame(rows)


# Show the verbose breakdown for a 3-leg and a 4-leg intersection
example_3leg = intersections[intersections["street_count"] == 3].index[0]
example_4leg_id = example_4leg  # defined above

print("=== 3-leg intersection ===")
df3 = get_leg_bearings_verbose(example_3leg, wegvakken, intersections)
print(df3.to_string(index=False))

print("\n=== 4-leg intersection ===")
df4 = get_leg_bearings_verbose(example_4leg_id, wegvakken, intersections)
print(df4.to_string(index=False))

In [ ]:
# Visualise the leg bearings as arrows radiating out from the intersection center
# This makes the geometric meaning of 'approach bearing' immediately clear.

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, jte_id, df_legs, title in [
    (axes[0], example_3leg,   df3, "3-leg intersection"),
    (axes[1], example_4leg_id, df4, "4-leg intersection"),
]:
    ix = intersections.loc[jte_id].geometry.x
    iy = intersections.loc[jte_id].geometry.y
    buf = 120

    # Background: nearby road segments
    nearby = wegvakken.cx[ix - buf:ix + buf, iy - buf:iy + buf]
    nearby.plot(ax=ax, color="lightgrey", linewidth=1.5)

    # Intersection centroid
    ax.scatter(ix, iy, color="red", s=120, zorder=5)

    # Arrows showing approach bearings â€” one per non-duplicate leg
    kept = df_legs[~df_legs["is_duplicate"]]
    for _, leg in kept.iterrows():
        bearing_rad = np.radians(leg["approach_bearing"])
        # Arrow points FROM 80m away (in approach direction) TO intersection
        # bearing 0=North so: dx=sin(b)*r, dy=cos(b)*r
        r = 80
        start_x = ix - np.sin(bearing_rad) * r
        start_y = iy - np.cos(bearing_rad) * r
        ax.annotate(
            f"{leg['approach_bearing']:.0f}Â°",
            xy=(ix, iy), xytext=(start_x, start_y),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=2),
            fontsize=9, color="steelblue", ha="center"
        )

    # Duplicates (two-way roads counted twice) shown as dashed
    dups = df_legs[df_legs["is_duplicate"]]
    for _, leg in dups.iterrows():
        bearing_rad = np.radians(leg["approach_bearing"])
        r = 80
        start_x = ix - np.sin(bearing_rad) * r
        start_y = iy - np.cos(bearing_rad) * r
        ax.annotate(
            f"{leg['approach_bearing']:.0f}Â° (dup)",
            xy=(ix, iy), xytext=(start_x, start_y),
            arrowprops=dict(arrowstyle="->", color="lightcoral", lw=1.5, linestyle="dashed"),
            fontsize=8, color="lightcoral", ha="center"
        )

    ax.set_xlim(ix - buf, ix + buf)
    ax.set_ylim(iy - buf, iy + buf)
    ax.set_aspect("equal")
    ax.set_title(f"{title}\nJTE_ID={jte_id}")

from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="steelblue", lw=2, label="Kept approach bearing"),
    Line2D([0], [0], color="lightcoral", lw=1.5, linestyle="dashed",
           label="Duplicate bearing (skipped)"),
]
axes[1].legend(handles=legend_elements, loc="upper right")

plt.suptitle("Approach bearings extracted from NWB\n(blue arrows = approach direction toward intersection)",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Potential issues with the current selection

### 7a. Dual carriageways

A dual-carriageway road has **two parallel wegvakken** â€” one per direction.
At a large junction, these may connect via separate junction IDs, causing the
pipeline to create **two intersection points very close together** instead of one.

Let's look for pairs of junctions that are very close to each other.

In [ ]:
# Find pairs of intersections that are very close together â€” potential dual carriageways
# Strategy: buffer each intersection by a small radius and find overlaps

# Use the intersections inside the ring
inter_ring = gpd.read_file(INTERSECTIONS_PATH).set_index("JTE_ID")

# Create small 15m buffers and check for overlaps
inter_buffered = inter_ring.copy()
inter_buffered["geometry"] = inter_buffered.buffer(15)  # 15m radius

# Spatial join of buffered intersections onto themselves
# left_index and right_index will be the same when they overlap themselves,
# so filter those out
overlaps = gpd.sjoin(
    inter_buffered.reset_index()[["JTE_ID", "geometry"]],
    inter_buffered.reset_index()[["JTE_ID", "geometry"]],
    how="inner",
    predicate="intersects"
)

# Remove self-matches (same JTE_ID)
overlaps = overlaps[overlaps["JTE_ID_left"] != overlaps["JTE_ID_right"]]

# Each pair appears twice (Aâ€“B and Bâ€“A), keep one direction
overlaps = overlaps[overlaps["JTE_ID_left"] < overlaps["JTE_ID_right"]]

print(f"Intersection pairs within 15m of each other: {len(overlaps):,}")
print("These may represent dual-carriageway junctions split into two points.")
print(f"\nAs a fraction of all intersections: {len(overlaps)*2/len(inter_ring)*100:.1f}%")

In [ ]:
# Visualise one example of a close pair
if len(overlaps) > 0:
    # Pick the first close pair
    pair = overlaps.iloc[0]
    jte_a, jte_b = pair["JTE_ID_left"], pair["JTE_ID_right"]

    ix_a = inter_ring.loc[jte_a].geometry.x
    iy_a = inter_ring.loc[jte_a].geometry.y
    ix_b = inter_ring.loc[jte_b].geometry.x
    iy_b = inter_ring.loc[jte_b].geometry.y

    dist_ab = np.sqrt((ix_a - ix_b)**2 + (iy_a - iy_b)**2)
    cx = (ix_a + ix_b) / 2
    cy = (iy_a + iy_b) / 2
    buf = 120

    fig, ax = plt.subplots(figsize=(8, 8))
    nearby = wegvakken.cx[cx - buf:cx + buf, cy - buf:cy + buf]
    nearby.plot(ax=ax, color="lightgrey", linewidth=1.5)

    ax.scatter(ix_a, iy_a, color="steelblue", s=120, zorder=5,
               label=f"JTE {jte_a} (sc={inter_ring.loc[jte_a, 'street_count']})")
    ax.scatter(ix_b, iy_b, color="darkorange", s=120, zorder=5,
               label=f"JTE {jte_b} (sc={inter_ring.loc[jte_b, 'street_count']})")

    ax.set_xlim(cx - buf, cx + buf)
    ax.set_ylim(cy - buf, cy + buf)
    ax.set_aspect("equal")
    ax.legend()
    ax.set_title(f"Example close pair â€” {dist_ab:.1f}m apart\n"
                 "Likely a dual-carriageway or split intersection")
    plt.tight_layout()
    plt.show()
else:
    print("No close pairs found at 15m threshold.")

### 7b. Ramps (AFR/OPR) inflating street_count

`AFR` (afrit = exit ramp) and `OPR` (oprit = entrance ramp) are currently included in the pipeline.
This means a motorway junction with two ramps can register as an intersection with `street_count >= 3`
even though it is not a standard at-grade intersection.

Let's see how many intersections would disappear if ramps were excluded.

In [ ]:
# Recalculate street_count WITHOUT ramps to see the effect
NO_RAMP_BST = INCLUDE_BST - {"AFR", "OPR"}

wegvakken_noramp = wegvakken_rot[wegvakken_rot["BST_CODE"].isin(NO_RAMP_BST)].copy()

beg_nr = wegvakken_noramp["JTE_ID_BEG"].value_counts()
end_nr  = wegvakken_noramp["JTE_ID_END"].value_counts()
cc_noramp = beg_nr.add(end_nr, fill_value=0).astype(int)

# How many junctions go from >= 3 to < 3 when ramps are removed?
with_ramp    = set(connection_counts[connection_counts >= 3].index)
without_ramp = set(cc_noramp[cc_noramp >= 3].index)

lost = with_ramp - without_ramp
print(f"Intersections with ramps (AFR/OPR) included:  {len(with_ramp):,}")
print(f"Intersections without ramps (AFR/OPR):        {len(without_ramp):,}")
print(f"Junctions that LOSE intersection status:      {len(lost):,}")
print()
print("These are likely motorway/slip road junctions that only qualify")
print("as intersections because of the ramp count.")

### 7c. How does the duplicate-bearing filter (20Â° threshold) affect leg counts?

Two-way roads in NWB appear as a single wegvak traversed in both directions,
but one-way roads may have a separate return wegvak. The `< 20Â°` deduplication
removes near-identical bearings that represent the same physical road.

Here we check how many legs are removed by this filter across a sample of intersections.

In [ ]:
# Count raw legs (before dedup) vs. kept legs (after dedup) for first 200 intersections
sample_ids = intersections.index[:200]

raw_counts  = []  # total connected wegvakken
kept_counts = []  # unique legs after dedup

for jte_id in sample_ids:
    node = intersections.loc[jte_id]
    ix, iy = node.geometry.x, node.geometry.y

    connected = wegvakken[
        (wegvakken["JTE_ID_BEG"] == jte_id) |
        (wegvakken["JTE_ID_END"] == jte_id)
    ]
    raw_counts.append(len(connected))

    # Apply the same dedup logic as notebook 04
    seen_bearings = []
    kept = 0
    for _, wvk in connected.iterrows():
        if wvk["JTE_ID_BEG"] == jte_id:
            nx, ny = wvk.geometry.coords[-1]
        else:
            nx, ny = wvk.geometry.coords[0]
        b = bearing_between(nx, ny, ix, iy)
        if not any(abs((b - s + 180) % 360 - 180) < 20 for s in seen_bearings):
            seen_bearings.append(b)
            kept += 1
    kept_counts.append(kept)

dedup_df = pd.DataFrame({"raw": raw_counts, "kept": kept_counts})
dedup_df["removed"] = dedup_df["raw"] - dedup_df["kept"]

print("Leg deduplication summary (sample of 200 intersections):")
print(dedup_df[["raw", "kept", "removed"]].describe().round(2))

print(f"\nTotal raw legs:    {dedup_df['raw'].sum():,}")
print(f"Total kept legs:   {dedup_df['kept'].sum():,}")
print(f"Removed as dups:   {dedup_df['removed'].sum():,} "
      f"({dedup_df['removed'].sum()/dedup_df['raw'].sum()*100:.1f}%)")

In [ ]:
# Scatter: raw leg count vs kept leg count per intersection
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(dedup_df["raw"], dedup_df["kept"], alpha=0.4, s=20, color="steelblue")
ax.plot([0, dedup_df["raw"].max()], [0, dedup_df["raw"].max()],
        color="black", linestyle="--", linewidth=1, label="No deduplication (y=x)")
ax.set_xlabel("Raw leg count (all connected wegvakken)")
ax.set_ylabel("Kept leg count (after 20Â° dedup)")
ax.set_title("Effect of bearing deduplication per intersection")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. What's in `Hectopunten.gpkg`? (currently unused)

The `Hectopunten` file is in the raw data folder but is not used by the current pipeline.
Here we quickly inspect it to understand what it contains.

In [ ]:
hecto_path = os.path.join(PROJECT_DIR, "data", "raw", "Hectopunten.gpkg")

try:
    hecto = gpd.read_file(hecto_path)
    print(f"Hectopunten rows: {len(hecto):,}")
    print(f"Columns: {list(hecto.columns)}")
    print()
    print(hecto.drop(columns="geometry").head(5).to_string())
except Exception as e:
    print(f"Could not load Hectopunten: {e}")

---
## 9. Kunnen we NWB-juncties direct gebruiken in plaats van `street_count`?

De NWB-documentatie beschrijft juncties als expliciete objecten. Maar de `Wegvakken.gpkg` bevat slechts **één layer** — geen aparte Juncties-laag. Juncties zijn impliciet opgeslagen als `JTE_ID_BEG`/`JTE_ID_END` in de wegvakken.

NDW levert een aparte `Juncties.gpkg` als extra download, maar die ontbreekt hier.

**Zelfs een aparte Juncties-laag zou nog steeds filtering nodig hebben**, want de NWB-documentatie zegt expliciet dat 2-verbindingsjuncties ook voorkomen bij:
- Straatnaamwisselingen
- Gemeentegrenzen
- Wegbeheerderswisselingen
- Doodlopende wegen (keerlus)

De huidige aanpak (`street_count >= 3` afleiden uit wegvakeindpunten) is dus **equivalent aan wat een aparte Juncties-laag zou geven**, maar dan met de juiste filter er al op.

In [ ]:
# Confirm: only one layer in the GPKG — no separate Juncties layer
import pyogrio
layers = pyogrio.list_layers(WEGVAKKEN_PATH)
print(f'Layers in Wegvakken.gpkg: {layers}')
print()
# Sanity check: junction coordinates from intersections.gpkg
# should match the wegvak endpoints exactly (or within float precision)
intersections_check = gpd.read_file(INTERSECTIONS_PATH).set_index('JTE_ID')
print('Spot-check: distance between saved junction and wegvak endpoint:')
for jte_id in intersections_check.index[:5]:
    ix = intersections_check.loc[jte_id].geometry.x
    iy = intersections_check.loc[jte_id].geometry.y
    # Try BEG first, then END
    wvk_beg = wegvakken[wegvakken['JTE_ID_BEG'] == jte_id]
    if len(wvk_beg) > 0:
        ex, ey = wvk_beg.iloc[0].geometry.coords[0]
    else:
        wvk_end = wegvakken[wegvakken['JTE_ID_END'] == jte_id]
        ex, ey = wvk_end.iloc[0].geometry.coords[-1]
    dist = ((ix - ex)**2 + (iy - ey)**2)**0.5
    print(f'  JTE {jte_id}: {dist:.4f}m')
print()
print('Near-zero distances confirm: pipeline extracts the correct junction locations.')

---
## 10. `RIJRICHTNG` — rijden auto's daadwerkelijk naar de kruising toe?

De huidige pipeline selecteert legs puur op geometrie. Maar `RIJRICHTNG` bepaalt of verkeer een wegvak in een bepaalde richting mag:

- `B` = Beide richtingen
- `H` = Heen (BEG naar END)
- `T` = Terug (END naar BEG)

| Junctie is... | RIJRICHTNG | Verkeer richting junctie? |
|---|---|---|
| `JTE_ID_END` | `H` | Ja — BEG→END stroomt naar de junctie |
| `JTE_ID_END` | `T` | **Nee** — END→BEG stroomt van de junctie af |
| `JTE_ID_BEG` | `H` | **Nee** — BEG→END stroomt van de junctie af |
| `JTE_ID_BEG` | `T` | Ja — END→BEG stroomt naar de junctie |
| beide | `B` | Ja — tweerichtingsverkeer |

In [ ]:
# Distribution of RIJRICHTNG in filtered Rotterdam wegvakken
rij_dist = wegvakken['RIJRICHTNG'].value_counts()
print('RIJRICHTNG distribution (Rotterdam, BST_CODE filtered):')
for val, count in rij_dist.items():
    print(f'  {val}: {count:,} ({count/len(wegvakken)*100:.1f}%)')

In [ ]:
# For 200 intersections: how many legs would a RIJRICHTNG filter remove?
inter_rij = gpd.read_file(INTERSECTIONS_PATH).set_index('JTE_ID')
legs_before, legs_after = 0, 0
removed_examples = []

for jte_id in inter_rij.index[:200]:
    connected = wegvakken[
        (wegvakken['JTE_ID_BEG'] == jte_id) |
        (wegvakken['JTE_ID_END'] == jte_id)
    ]
    legs_before += len(connected)
    for _, wvk in connected.iterrows():
        rij = wvk['RIJRICHTNG']
        at_end = (wvk['JTE_ID_END'] == jte_id)
        # Traffic approaches the junction if:
        # B: both directions, H: traffic goes BEG->END so valid if junction is END,
        # T: traffic goes END->BEG so valid if junction is BEG
        can_approach = (
            rij == 'B'
            or (rij == 'H' and at_end)
            or (rij == 'T' and not at_end)
        )
        if can_approach:
            legs_after += 1
        elif len(removed_examples) < 5:
            removed_examples.append({
                'JTE_ID': jte_id, 'WVK_ID': wvk['WVK_ID'],
                'BST_CODE': wvk['BST_CODE'], 'RIJRICHTNG': rij,
                'role': 'END' if at_end else 'BEG',
                'STT_NAAM': wvk['STT_NAAM']
            })

removed = legs_before - legs_after
print('Sample of 200 intersections:')
print(f'  Legs before RIJRICHTNG filter: {legs_before:,}')
print(f'  Legs after  RIJRICHTNG filter: {legs_after:,}')
print(f'  Removed (one-way exit legs):   {removed:,} ({removed/legs_before*100:.1f}%)')
print()
print('Examples of removed (exit-only) legs:')
print(pd.DataFrame(removed_examples).to_string(index=False))

In [ ]:
# Visualise a concrete example: intersection with a one-way exit leg
inter_vis = gpd.read_file(INTERSECTIONS_PATH).set_index('JTE_ID')
example_oneway = None
for jte_id in inter_vis.index[:500]:
    connected = wegvakken[
        (wegvakken['JTE_ID_BEG'] == jte_id) |
        (wegvakken['JTE_ID_END'] == jte_id)
    ]
    for _, wvk in connected.iterrows():
        at_end = (wvk['JTE_ID_END'] == jte_id)
        rij = wvk['RIJRICHTNG']
        can_approach = (
            rij == 'B' or (rij == 'H' and at_end) or (rij == 'T' and not at_end)
        )
        if not can_approach:
            example_oneway = jte_id
            break
    if example_oneway:
        break

if example_oneway:
    ix = inter_vis.loc[example_oneway].geometry.x
    iy = inter_vis.loc[example_oneway].geometry.y
    buf = 130
    conn = wegvakken[
        (wegvakken['JTE_ID_BEG'] == example_oneway) |
        (wegvakken['JTE_ID_END'] == example_oneway)
    ]
    fig, ax = plt.subplots(figsize=(9, 9))
    wegvakken.cx[ix-buf:ix+buf, iy-buf:iy+buf].plot(ax=ax, color='lightgrey', lw=1)
    ax.scatter(ix, iy, color='red', s=150, zorder=5)
    from matplotlib.lines import Line2D
    for _, wvk in conn.iterrows():
        at_end = (wvk['JTE_ID_END'] == example_oneway)
        rij = wvk['RIJRICHTNG']
        can_approach = (
            rij == 'B' or (rij == 'H' and at_end) or (rij == 'T' and not at_end)
        )
        c = 'steelblue' if can_approach else 'lightcoral'
        # Draw the wegvak
        gpd.GeoDataFrame([wvk], geometry='geometry', crs=CRS_RD).plot(
            ax=ax, color=c, lw=3)
        # Draw an arrow showing the allowed traffic direction
        coords = list(wvk.geometry.coords)
        mid = max(1, len(coords) // 2)
        x0, y0 = coords[mid - 1]
        x1, y1 = coords[mid]
        if rij == 'T':   # traffic flows END->BEG, so reverse the arrow
            x0, y0, x1, y1 = x1, y1, x0, y0
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2.5))
    ax.legend(handles=[
        Line2D([0],[0], color='steelblue', lw=3, label='Approach leg (valid)'),
        Line2D([0],[0], color='lightcoral', lw=3,
               label='Exit-only leg (current pipeline treats as approach)'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
               ms=10, label='Junction'),
    ])
    ax.set_xlim(ix - buf, ix + buf)
    ax.set_ylim(iy - buf, iy + buf)
    ax.set_aspect('equal')
    ax.set_title(
        f'RIJRICHTNG example  JTE_ID={example_oneway}\n'
        'Red = one-way exit: current pipeline incorrectly looks for an approach photo here'
    )
    plt.tight_layout()
    plt.show()
else:
    print('No one-way exit example found in first 500 intersections.')

---
## 11. `FOW` — Form of Way: rotondes, gescheiden rijbanen, afritten

| Code | Omschrijving | Relevant? |
|---|---|---|
| 1 | Motorway | Ja |
| 2 | Multiple carriageway | **Ja** — gescheiden rijbanen leiden tot split juncties |
| 3 | Single carriageway | Standaard |
| 4 | Roundabout | **Ja** — visueel heel anders dan gewone kruising |
| 6 | Sliproad | Ja — deels al gefilterd via BST_CODE |
| 7 | Other | Overig |

In [ ]:
# FOW distribution in filtered Rotterdam wegvakken
fow_labels = {
    '1': 'Motorway',
    '2': 'Multiple carriageway',
    '3': 'Single carriageway',
    '4': 'Roundabout',
    '6': 'Sliproad',
    '7': 'Other'
}
fow_dist = wegvakken['FOW'].astype(str).value_counts().sort_index()
fow_df = fow_dist.reset_index()
fow_df.columns = ['FOW', 'count']
fow_df['label'] = fow_df['FOW'].map(fow_labels).fillna('?')
fow_df['pct'] = (fow_df['count'] / len(wegvakken) * 100).round(1)
print('FOW distribution (Rotterdam, BST_CODE filtered):')
print(fow_df.to_string(index=False))

In [ ]:
# Count intersections with roundabout (FOW=4) or multiple-carriageway (FOW=2) legs
inter_fow = gpd.read_file(INTERSECTIONS_PATH).set_index('JTE_ID')
roundabout_ids, mc_ids, slip_ids = [], [], []

for jte_id in inter_fow.index:
    connected = wegvakken[
        (wegvakken['JTE_ID_BEG'] == jte_id) |
        (wegvakken['JTE_ID_END'] == jte_id)
    ]
    fow_vals = set(connected['FOW'].dropna().astype(str).values)
    if '4' in fow_vals:
        roundabout_ids.append(jte_id)
    if '2' in fow_vals:
        mc_ids.append(jte_id)
    if '6' in fow_vals:
        slip_ids.append(jte_id)

n = len(inter_fow)
print(f'FOW=4 roundabout:            {len(roundabout_ids):,} ({len(roundabout_ids)/n*100:.1f}%)')
print(f'FOW=2 multiple carriageway:  {len(mc_ids):,} ({len(mc_ids)/n*100:.1f}%)')
print(f'FOW=6 sliproad:              {len(slip_ids):,} ({len(slip_ids)/n*100:.1f}%)')

In [ ]:
# Map: roundabouts and multiple-carriageway intersections
ring_fow = gpd.read_file(RING_PATH).to_crs(CRS_RD)
roundabout_gdf = inter_fow.loc[roundabout_ids]
mc_gdf = inter_fow.loc[mc_ids]
other_ids = list(set(inter_fow.index) - set(roundabout_ids) - set(mc_ids))
other_gdf = inter_fow.loc[other_ids]

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, gdf, title, color in [
    (axes[0], roundabout_gdf,
     f'FOW=4 Roundabouts ({len(roundabout_gdf):,})', 'darkorange'),
    (axes[1], mc_gdf,
     f'FOW=2 Multiple carriageway ({len(mc_gdf):,})', 'purple'),
]:
    other_gdf.plot(ax=ax, color='lightgrey', markersize=2, alpha=0.5)
    gdf.plot(ax=ax, color=color, markersize=10, alpha=0.85)
    ring_fow.plot(ax=ax, color='none', edgecolor='red', linewidth=1.5)
    ax.set_aspect('equal')
    ax.set_title(title)
plt.suptitle('FOW-based intersection classification', fontsize=12)
plt.tight_layout()
plt.show()

---
## 12. `FRC` — Functional Road Class

`FRC` loopt van 0 (snelwegen) tot 7 (niet voor autoverkeer). Controleert of er FRC=7 segmenten door de BST_CODE filter glippen.

In [ ]:
# FRC distribution in filtered Rotterdam wegvakken
frc_labels = {
    '0': 'A-wegen hoofdrijbaan',
    '1': 'Parallelbanen A + rijkswegen N',
    '2': 'Overige rijkswegen + prov. N<400',
    '3': 'Overige provinciale wegen',
    '4': 'Overige wegen >50 km/h',
    '5': 'Overige wegen 50 km/h',
    '6': 'Overige wegen voor autoverkeer',
    '7': 'NIET voor autoverkeer'
}
frc_dist = wegvakken['FRC'].astype(str).value_counts().sort_index()
frc_df = frc_dist.reset_index()
frc_df.columns = ['FRC', 'count']
frc_df['label'] = frc_df['FRC'].map(frc_labels).fillna('?')
frc_df['pct'] = (frc_df['count'] / len(wegvakken) * 100).round(1)
print('FRC distribution (Rotterdam, BST_CODE filtered):')
print(frc_df.to_string(index=False))
print()
# How many FRC=7 (not for car traffic) slipped through the BST_CODE filter?
frc7 = wegvakken[wegvakken['FRC'].astype(str) == '7']
print(f'FRC=7 that passed BST_CODE filter: {len(frc7):,}')
if len(frc7) > 0:
    print('BST_CODE breakdown of these:')
    print(frc7['BST_CODE'].value_counts().to_string())

---
## 13. Administratieve splitsingen — telt `street_count` soms twee helften van dezelfde weg?

Bij straatnaamwisselingen en gemeentegrenzen plaatst het NWB een extra junctie, ook als de weg fysiek rechtdoor loopt. Bij een `street_count=3` kruising kunnen dan 2 van de 3 wegvakken dezelfde straatnaam hebben.

Dit is nog steeds een echte T-splitsing. Maar de leg-extractie produceert twee bijna-identieke bearings voor de doorgaande weg. De 20 graden dedup-filter in notebook 04 vangt dit doorgaans correct op.

In [ ]:
from collections import Counter

inter_adm = gpd.read_file(INTERSECTIONS_PATH).set_index('JTE_ID')
inter_3leg = inter_adm[inter_adm['street_count'] == 3]

admin_split_count = 0
admin_split_examples = []

for jte_id in inter_3leg.index:
    connected = wegvakken[
        (wegvakken['JTE_ID_BEG'] == jte_id) |
        (wegvakken['JTE_ID_END'] == jte_id)
    ]
    names = connected['STT_NAAM'].dropna().values
    name_counts = Counter(names)
    # A repeated street name means the through-road was split by an admin boundary
    has_repeated = any(v >= 2 for v in name_counts.values())
    if has_repeated:
        admin_split_count += 1
        if len(admin_split_examples) < 5:
            repeated = [nm for nm, c in name_counts.items() if c >= 2]
            admin_split_examples.append({
                'JTE_ID': jte_id,
                'street_names': list(names),
                'repeated': repeated[0] if repeated else ''
            })

print(f'street_count=3 intersections in ring:      {len(inter_3leg):,}')
print(f'With 2+ wegvakken sharing a street name:   {admin_split_count:,}'
      f' ({admin_split_count/len(inter_3leg)*100:.1f}%)')
print()
print('These are real T-junctions. The 20-degree dedup filter handles the')
print('near-duplicate bearings produced by the split through-road.')
print()
print('Examples:')
for ex in admin_split_examples:
    print(f'  JTE {ex["JTE_ID"]}: names={ex["street_names"]}'
          f', repeated="{ex["repeated"]}"')

---
## 14. Filterstrategieen vergelijken: BST_CODE vs. WEGBEHSRT='G' vs. combinatie

Notebook 00 heeft nu een schakelaar `USE_GEMEENTE_FILTER`. Hier vergelijken we de drie opties zodat je de impact kunt beoordelen:

| Strategie | Wat het doet |
|---|---|
| **A — BST_CODE only** (oud) | Filtert op wegtype; snelwegen (HR/AFR/OPR van Rijk) komen erdoor |
| **B — WEGBEHSRT='G' only** | Filtert op beheerder; fietspaden en parkeerplaatsen komen erdoor |
| **C — WEGBEHSRT='G' + BST_CODE** (nieuw) | Combineert beide; schoonste resultaat |

In [ ]:
# Load Rotterdam wegvakken once — used for all three strategies
print('Loading wegvakken...')
wvk_all = gpd.read_file(WEGVAKKEN_PATH)
wvk_rot = wvk_all[wvk_all['GME_NAAM'] == 'Rotterdam'].copy()
print(f'Rotterdam total: {len(wvk_rot):,} segments')

INCLUDE_BST = {'RB', 'ERF', 'HR'}  # normal roads only

# Strategy A: BST_CODE only (current pipeline)
wvk_A = wvk_rot[wvk_rot['BST_CODE'].isin(INCLUDE_BST)].copy()

# Strategy B: WEGBEHSRT='G' only (no BST_CODE filter)
wvk_B = wvk_rot[wvk_rot['WEGBEHSRT'] == 'G'].copy()

# Strategy C: WEGBEHSRT='G' + BST_CODE (recommended combination)
wvk_C = wvk_rot[
    (wvk_rot['WEGBEHSRT'] == 'G') &
    (wvk_rot['BST_CODE'].isin(INCLUDE_BST))
].copy()

print(f'Strategy A (BST_CODE only):        {len(wvk_A):,} segments')
print(f'Strategy B (WEGBEHSRT=G only):     {len(wvk_B):,} segments')
print(f'Strategy C (WEGBEHSRT=G + BST):    {len(wvk_C):,} segments')

In [ ]:
# Show what each strategy includes/excludes by WEGBEHSRT and BST_CODE
print('=== Strategy A — WEGBEHSRT breakdown (who manages these roads?) ===')
print(wvk_A['WEGBEHSRT'].value_counts().to_string())
print()
print('=== Strategy B — BST_CODE breakdown (what road types are included?) ===')
print(wvk_B['BST_CODE'].value_counts().to_string())
print()
print('=== Strategy C — WEGBEHSRT + BST_CODE breakdowns ===')
print('WEGBEHSRT:', wvk_C['WEGBEHSRT'].value_counts().to_string())
print('BST_CODE: ', wvk_C['BST_CODE'].value_counts().to_string())

In [ ]:
# Helper: derive intersections from a wegvakken GeoDataFrame
# Reproduces the logic from notebook 00 (extract junction endpoints, count connections)
from shapely.geometry import Point

def derive_intersections(wvk_gdf, min_connections=3):
    """Return a GeoDataFrame of junctions with street_count >= min_connections."""
    starts = wvk_gdf[['JTE_ID_BEG', 'geometry']].copy()
    starts['pt'] = starts['geometry'].apply(lambda g: Point(g.coords[0]))
    starts = starts.rename(columns={'JTE_ID_BEG': 'JTE_ID'})[['JTE_ID', 'pt']]

    ends = wvk_gdf[['JTE_ID_END', 'geometry']].copy()
    ends['pt'] = ends['geometry'].apply(lambda g: Point(g.coords[-1]))
    ends = ends.rename(columns={'JTE_ID_END': 'JTE_ID'})[['JTE_ID', 'pt']]

    junctions = pd.concat([starts, ends], ignore_index=True)
    junctions = junctions.drop_duplicates(subset='JTE_ID').set_index('JTE_ID')

    beg_cnt = wvk_gdf['JTE_ID_BEG'].value_counts()
    end_cnt = wvk_gdf['JTE_ID_END'].value_counts()
    junctions['street_count'] = beg_cnt.add(end_cnt, fill_value=0).astype(int)

    inter = junctions[junctions['street_count'] >= min_connections].reset_index()
    return gpd.GeoDataFrame(inter, geometry='pt', crs=CRS_RD)


# Derive intersections for each strategy and clip to the ring
ring_cmp = gpd.read_file(RING_PATH).to_crs(CRS_RD)
ring_poly = ring_cmp.union_all()

print('Deriving intersections for each strategy...')
inter_A = derive_intersections(wvk_A)
inter_A = inter_A[inter_A.within(ring_poly)]
print(f'Strategy A: {len(inter_A):,} intersections inside ring')

inter_B = derive_intersections(wvk_B)
inter_B = inter_B[inter_B.within(ring_poly)]
print(f'Strategy B: {len(inter_B):,} intersections inside ring')

inter_C = derive_intersections(wvk_C)
inter_C = inter_C[inter_C.within(ring_poly)]
print(f'Strategy C: {len(inter_C):,} intersections inside ring')

In [ ]:
# Which intersections appear in A but NOT in C?
# These are the junctions that the gemeente filter removes — likely motorway junctions.
ids_A = set(inter_A['JTE_ID'])
ids_C = set(inter_C['JTE_ID'])

only_in_A = inter_A[inter_A['JTE_ID'].isin(ids_A - ids_C)].copy()
only_in_C = inter_C[inter_C['JTE_ID'].isin(ids_C - ids_A)].copy()

print(f'In A but not C (removed by gemeente filter): {len(only_in_A):,}')
print(f'In C but not A (added by gemeente filter):   {len(only_in_C):,}')
print()
print('street_count distribution of removed intersections (A only):')
print(only_in_A['street_count'].value_counts().sort_index().to_string())

In [ ]:
# Map: side-by-side comparison of strategy A vs strategy C
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Left: what strategy A has that C does not (removed by gemeente filter)
ax = axes[0]
inter_A[inter_A['JTE_ID'].isin(ids_A & ids_C)].plot(
    ax=ax, color='steelblue', markersize=2, alpha=0.5, label='In both A and C')
only_in_A.plot(ax=ax, color='lightcoral', markersize=6, alpha=0.9,
               label=f'Only in A — removed by gemeente filter ({len(only_in_A):,})')
ring_cmp.plot(ax=ax, color='none', edgecolor='red', lw=1.5)
ax.set_aspect('equal')
ax.legend(markerscale=3, fontsize=8)
ax.set_title(f'Strategy A (BST_CODE only)\n{len(inter_A):,} intersections')

# Right: strategy C (gemeente + BST_CODE)
ax = axes[1]
inter_C.plot(ax=ax, color='steelblue', markersize=2, alpha=0.6)
ring_cmp.plot(ax=ax, color='none', edgecolor='red', lw=1.5)
ax.set_aspect('equal')
ax.set_title(f'Strategy C (WEGBEHSRT=G + BST_CODE)\n{len(inter_C):,} intersections')

plt.suptitle('Filter strategy comparison — red dots are motorway/provincial junctions'
             ' excluded by Strategy C', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on one removed intersection to confirm it is a motorway junction
if len(only_in_A) > 0:
    # Pick the one with the highest street_count — most likely a complex junction
    ex_id = only_in_A.sort_values('street_count', ascending=False).iloc[0]['JTE_ID']
    ex_row = only_in_A[only_in_A['JTE_ID'] == ex_id].iloc[0]
    ix, iy = ex_row.geometry.x, ex_row.geometry.y
    buf = 200

    # Find the wegvakken connected to this junction in strategy A
    conn_A = wvk_A[(wvk_A['JTE_ID_BEG'] == ex_id) | (wvk_A['JTE_ID_END'] == ex_id)]

    fig, ax = plt.subplots(figsize=(9, 9))
    # Background: all gemeente wegvakken nearby
    wvk_C.cx[ix-buf:ix+buf, iy-buf:iy+buf].plot(
        ax=ax, color='lightgrey', lw=1.5, label='Gemeente roads (C)')
    # The non-gemeente wegvakken that A includes but C excludes
    non_gem = conn_A[conn_A['WEGBEHSRT'] != 'G']
    gem_conn = conn_A[conn_A['WEGBEHSRT'] == 'G']
    gem_conn.plot(ax=ax, color='steelblue', lw=3, label='Gemeente legs')
    non_gem.plot(ax=ax, color='lightcoral', lw=3,
                 label=f'Non-gemeente legs (WEGBEHSRT={non_gem["WEGBEHSRT"].unique()})')
    ax.scatter(ix, iy, color='red', s=150, zorder=5, label='Junction')
    ax.set_xlim(ix-buf, ix+buf); ax.set_ylim(iy-buf, iy+buf)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.set_title(
        f'Example removed junction — JTE_ID={ex_id}\n'
        f'street_count(A)={int(ex_row["street_count"])} | '
        f'WEGBEHSRT of connected roads: {conn_A["WEGBEHSRT"].value_counts().to_dict()}'
    )
    plt.tight_layout()
    plt.show()
    print('Connected wegvakken at this junction:')
    print(conn_A[['WVK_ID','BST_CODE','WEGBEHSRT','FOW','STT_NAAM']].to_string(index=False))

---
## Bijgewerkt overzicht van bevindingen

| Aspect | Bevinding |
|---|---|
| NWB dekkingsgraad | ~1,6M wegvakken nationaal; ~27k Rotterdam na BST_CODE filter |
| Junctie-encoding | Impliciet via `JTE_ID_BEG`/`JTE_ID_END` — geen aparte Juncties-laag |
| `street_count` aanpak | Correct en equivalent aan een aparte Juncties-laag met filter |
| Intersectie-drempel | `street_count >= 3` vangt T-splitsingen en groter |
| **`RIJRICHTNG`** | **Pipeline filtert eenrichtingswegen niet — exit-only legs worden als approach-leg meegenomen** |
| **`FOW=4` (rotonde)** | **Rotondes worden niet geflagd; visueel anders dan reguliere kruisingen** |
| **`FOW=2` (gescheiden rijbanen)** | **Verklaart gesplitste junctieparen: 1 fysieke kruising = 2-4 NWB-juncties** |
| **`FRC=7`** | **Controleer of niet-autoverkeer segmenten door de BST_CODE filter glippen** |
| Admin. splitsingen | Doorgaande weg gesplitst op naamgrens; 20-graden dedup vangt dit op |
| Dubbele rijbanen | Close pairs (<15m) wijzen op 1 fysieke kruising in twee NWB-juncties |